# Data Download Automation for SDS Forecasting
Downloads the data types from the guide into one organised `./data/` folder.

| Step | Data | Tool | Account needed |
|---|---|---|---|
| 1 | NWP forecasts (ECMWF open data) | `ecmwf-opendata` | No |
| 2 | ERA5 reanalysis | `cdsapi` | Free CDS account |
| 3 | Satellite: MODIS AOD, SMAP soil moisture | `earthaccess` | Free NASA Earthdata account |
| 4 | Ground stations | manual / provider portal | Depends on provider |
| 5 | Climate indices (AO, NAO, Nino 3.4) | `requests` | No |
| 6 | GIS: elevation (DEM) | `earthaccess` | Free NASA Earthdata account |
| 7 | Surveys | your own files | n/a |

**Run order:** cell 1 (install) → cell 2 (config) → any download section → the final check cell.

## 0. Install and set up

In [ ]:
# Run once. Restart the kernel afterwards if imports fail.
%pip install -q cdsapi earthaccess ecmwf-opendata xarray netCDF4 requests pandas tqdm

In [ ]:
from pathlib import Path
import os, datetime as dt

# ---- EDIT THESE ----
AREA   = dict(north=45, west=75, south=30, east=120)   # North China dust study area
START  = "2025-03-01"
END    = "2025-05-31"
YEARS  = ["2025"]
MONTHS = ["03", "04", "05"]
DATA   = Path("./data")
# --------------------

for sub in ["nwp", "era5", "satellite/modis_aod", "satellite/smap", "ground", "s2s", "gis", "survey"]:
    (DATA / sub).mkdir(parents=True, exist_ok=True)
print("Folders ready under", DATA.resolve())

### One-time credentials
**CDS (ERA5):** register at https://cds.climate.copernicus.eu , accept the ERA5 licence on the dataset page, then create the file `~/.cdsapirc` containing:
```
url: https://cds.climate.copernicus.eu/api
key: <your-personal-access-token>
```
**NASA Earthdata:** register at https://urs.earthdata.nasa.gov . Either set the environment variables `EARTHDATA_USERNAME` and `EARTHDATA_PASSWORD`, or let `earthaccess` prompt you.

## 1. NWP forecasts (ECMWF open data, no account)
Only recent forecasts are available here. For older dates use the ECMWF archive (MARS) or TIGGE.

In [ ]:
from ecmwf.opendata import Client

client = Client(source="ecmwf")      # mirrors: "aws", "azure", "google" also work
nwp_file = DATA / "nwp" / "ifs_ens_control_latest.grib2"

client.retrieve(
    stream="enfo", type="cf",                  # ensemble control run, reaches 15 days
    param=["10u", "10v", "2t", "msl"],          # wind u/v, temperature, pressure
    step=list(range(0, 361, 24)),               # lead time in hours (0-360 = 15 days)
    target=str(nwp_file),
)
print("Saved", nwp_file, round(nwp_file.stat().st_size/1e6, 1), "MB")

## 2. ERA5 reanalysis (`cdsapi`)

In [ ]:
import cdsapi
c = cdsapi.Client()

era5_file = DATA / "era5" / f"era5_dust_{YEARS[0]}_spring.nc"
if not era5_file.exists():
    c.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "variable": [
                "10m_u_component_of_wind", "10m_v_component_of_wind",
                "2m_temperature", "boundary_layer_height",
                "mean_sea_level_pressure", "volumetric_soil_water_layer_1",
            ],
            "year": YEARS,
            "month": MONTHS,
            "day": [f"{d:02d}" for d in range(1, 32)],
            "time": [f"{h:02d}:00" for h in range(0, 24, 3)],
            "area": [AREA["north"], AREA["west"], AREA["south"], AREA["east"]],  # N, W, S, E
            "data_format": "netcdf",
            "download_format": "unarchived",
        },
        str(era5_file),
    )
print("ERA5 ready:", era5_file)

## 3. Satellite data (`earthaccess`)

In [ ]:
import earthaccess
auth = earthaccess.login(strategy="all")   # env vars, then ~/.netrc, then interactive prompt
print("Logged in:", auth.authenticated)

In [ ]:
# MODIS combined MAIAC Aerosol Optical Depth (AOD)
bbox = (AREA["west"], AREA["south"], AREA["east"], AREA["north"])   # west, south, east, north

aod = earthaccess.search_data(short_name="MCD19A2", bounding_box=bbox, temporal=(START, END))
print(len(aod), "MODIS AOD granules found")
# Tip: this is large. Test on a short date range first, then run the full period.
aod_files = earthaccess.download(aod, str(DATA / "satellite" / "modis_aod"))

In [ ]:
# SMAP enhanced daily soil moisture (9 km)
smap = earthaccess.search_data(short_name="SPL3SMP_E", bounding_box=bbox, temporal=(START, END))
print(len(smap), "SMAP granules found")
smap_files = earthaccess.download(smap, str(DATA / "satellite" / "smap"))

## 4. Ground stations (manual step)
There is no single open API for these, so download them from the provider and place files in `./data/ground/`:
* **CMA station data:** National Meteorological Information Center (NMIC) data portal, usually needs registration.
* **CNEMC PM10 / PM2.5:** China National Environmental Monitoring Centre; many researchers use archived copies from open data repositories.
* Needed columns: station id, time (UTC), PM10, PM2.5, visibility, wind, gust, weather code WW.

## 5. Climate indices (AO, NAO, Nino 3.4) from NOAA CPC

In [ ]:
import requests, pandas as pd, io

def fetch(url):
    r = requests.get(url, timeout=60); r.raise_for_status(); return r.text

# Daily AO and NAO: columns are year, month, day, value
base = "https://ftp.cpc.ncep.noaa.gov/cwlinks/"
for name, fn in {"ao": "norm.daily.ao.index.b500101.current.ascii",
                 "nao": "norm.daily.nao.index.b500101.current.ascii"}.items():
    txt = fetch(base + fn)
    df = pd.read_csv(io.StringIO(txt), sep=r"\s+", names=["year","month","day",name])
    df["date"] = pd.to_datetime(df[["year","month","day"]])
    df[["date", name]].to_csv(DATA/"s2s"/f"{name}_daily.csv", index=False)
    print(name, df.date.min().date(), "->", df.date.max().date(), len(df), "rows")

# Monthly Nino 3.4 (ENSO)
txt = fetch("https://www.cpc.ncep.noaa.gov/data/indices/sstoi.indices")
(DATA/"s2s"/"sstoi_indices.txt").write_text(txt)
print("Nino indices saved (Nino 3.4 columns are in the file).")

## 6. GIS: elevation model (DEM)
Boundaries (NGCC), railways and land cover (ESA WorldCover) come from their portals; download those manually into `./data/gis/`.

In [ ]:
dem = earthaccess.search_data(short_name="NASADEM_HGT", bounding_box=bbox)
print(len(dem), "DEM tiles found")
dem_files = earthaccess.download(dem, str(DATA / "gis"))

## 7. Surveys
Put your `.sav` (SPSS) or Excel files in `./data/survey/`. Nothing to download.

## 8. Check what you got

In [ ]:
import xarray as xr

# Inspect ERA5
ds = xr.open_dataset(era5_file)
print(ds)
print("\nTime range:", str(ds.valid_time.min().values)[:10], "->", str(ds.valid_time.max().values)[:10])

# Inventory of everything downloaded
print("\nFile inventory")
for f in sorted(DATA.rglob("*")):
    if f.is_file():
        print(f"{f.relative_to(DATA)!s:70} {f.stat().st_size/1e6:8.1f} MB")

## Next step: harmonization
Now follow the 5 stages from the presentation: regrid to 0.125°, align to UTC, fill gaps, scale, and package into tensors.

**Troubleshooting**
* `401 / licence not accepted` for ERA5: open the dataset page on the CDS site and accept its terms.
* `earthaccess` finds 0 granules: check dates and that `bbox` order is (west, south, east, north).
* Downloads are huge: shorten `START`/`END` or `AREA` first.